# E09

Segment Anything Model 2 inference copy for the real-video transfer example.

This notebook is the transfer-inference step: it reads the multi-clip
``sam2-caustic-transfer-manifest.json`` written by the prompt-manifest notebook
and runs both the base SAM2 and the fine-tuned SAM2 checkpoint on every clip
in that manifest. Each clip writes its outputs to
``outputs/E09/inference_outputs/<clip_stem>/`` with separate ``base/`` and
``finetuned/`` variant folders. A batch manifest
``outputs/E09/inference_outputs/e09_batch_manifest.json`` records the
clip-level state for the paper-assembly script.

Set ``SYNISCOPY_E09_CLIP_INDEX`` (an integer or a comma-separated list) before
running to restrict the batch to a subset of clip indices, for example when
testing a single clip in Colab. Without that override the notebook runs every
clip in the transfer manifest. Reruns are robust: a clip/variant whose expected
output files (overlay AVI, mask-only AVI, ``mask_metrics.json``, and per-frame
PNG masks) all already exist and look complete is skipped.

The public starter notebook stays in ``sam2_starter/``.

In [ ]:
#1
!nvidia-smi

In [ ]:
#2: Mount Drive and select checkpoint/output labels
import os, re, shutil, json
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
HOME = '/content'
print('HOME:', HOME)

# E09 uses the E08 trained checkpoint but writes inference outputs under E09.
CHECKPOINT_LABEL = 'E08'
OUTPUT_LABEL = 'E09'

RELEASE_NOTEBOOK_VERSION = 'syniscopy-sam2-inference-public-release-2026-05-27'

def sanitize_label(label: str) -> str:
    clean = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(label).strip())
    return clean.strip('._-') or 'default_configuration'

CHECKPOINT_LABEL = sanitize_label(CHECKPOINT_LABEL)
OUTPUT_LABEL = sanitize_label(OUTPUT_LABEL)
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([Path('/content/supplemental'), Path('/content/SyniscopySupplemental')])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists():
            return resolved
    raise RuntimeError('Upload the supplemental folder to Drive as MyDrive/supplemental.')


SUPPLEMENTAL_ROOT = find_supplemental_root()
DRIVE_ROOT = SUPPLEMENTAL_ROOT
OUTPUT_ROOT = SUPPLEMENTAL_ROOT / 'outputs'
CHECKPOINT_CONFIG_DIR = OUTPUT_ROOT / CHECKPOINT_LABEL
OUTPUT_CONFIG_DIR = OUTPUT_ROOT / OUTPUT_LABEL
WEIGHTS_DIR = CHECKPOINT_CONFIG_DIR / 'weights'
FINAL_CHECKPOINT_PATH = WEIGHTS_DIR / 'final_checkpoint.pt'
CHECKPOINT_MANIFEST_PATH = WEIGHTS_DIR / 'checkpoint_manifest.json'
INFERENCE_OUTPUT_ROOT = OUTPUT_CONFIG_DIR / 'inference_outputs'
INFERENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# The prompt-manifest notebook writes the transfer manifest that this notebook
# consumes. This notebook requires that multi-clip manifest as its input.
E06_TRANSFER_MANIFEST_PATH = OUTPUT_ROOT / 'E06' / 'sam2-caustic-transfer-manifest.json'

SAM2_CONFIG_NAME = 'configs/sam2.1/sam2.1_hiera_l.yaml'
SAM2_BASE_CHECKPOINT_FILENAME = 'sam2.1_hiera_large.pt'
SAM2_BASE_CHECKPOINT_URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'

if CHECKPOINT_MANIFEST_PATH.exists():
    with open(CHECKPOINT_MANIFEST_PATH, 'r') as f:
        checkpoint_manifest = json.load(f)
    SAM2_CONFIG_NAME = checkpoint_manifest.get('sam2_config_name', SAM2_CONFIG_NAME)
    SAM2_BASE_CHECKPOINT_FILENAME = checkpoint_manifest.get('sam2_base_checkpoint_filename', SAM2_BASE_CHECKPOINT_FILENAME)
    SAM2_BASE_CHECKPOINT_URL = checkpoint_manifest.get('sam2_base_checkpoint_url', SAM2_BASE_CHECKPOINT_URL)
    print('Loaded checkpoint manifest:', CHECKPOINT_MANIFEST_PATH)
else:
    checkpoint_manifest = None
    print('No checkpoint manifest found; using SAM2.1 Large defaults.')

if not FINAL_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'No final checkpoint found for label {CHECKPOINT_LABEL!r}: {FINAL_CHECKPOINT_PATH}. '
        'Run supplemental/E08.ipynb first.'
    )

print('Notebook version:', RELEASE_NOTEBOOK_VERSION)
print('Checkpoint label:', CHECKPOINT_LABEL)
print('Output label:', OUTPUT_LABEL)
print('Using fine-tuned checkpoint:', FINAL_CHECKPOINT_PATH)
print('Inference output root:', INFERENCE_OUTPUT_ROOT)
print('Transfer manifest:', E06_TRANSFER_MANIFEST_PATH)
print('SAM2 config:', SAM2_CONFIG_NAME)
print('SAM2 base checkpoint:', SAM2_BASE_CHECKPOINT_FILENAME)


In [ ]:
#3: Clone/install SAM2
import os
import shutil
import subprocess
import sys
from pathlib import Path

SAM2_DIR = Path(HOME) / 'segment-anything-2'
SAM2_GIT_URL = 'https://github.com/facebookresearch/segment-anything-2.git'
SAM2_GIT_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
os.chdir(HOME)


def _current_git_commit(repo_dir: Path) -> str | None:
    try:
        out = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'],
            cwd=str(repo_dir),
            stderr=subprocess.DEVNULL,
            text=True,
        )
    except Exception:
        return None
    return out.strip() or None


repo_valid = (
    (SAM2_DIR / 'setup.py').exists()
    and (SAM2_DIR / 'sam2').is_dir()
    and _current_git_commit(SAM2_DIR) == SAM2_GIT_COMMIT
)
if repo_valid:
    print(f'Reusing pinned SAM2 repository: {SAM2_DIR} @ {SAM2_GIT_COMMIT[:12]}')
else:
    if SAM2_DIR.exists():
        print('Removing SAM2 directory that is missing files or not at the pinned commit...')
        shutil.rmtree(SAM2_DIR)
    subprocess.run(['git', 'clone', SAM2_GIT_URL, str(SAM2_DIR)], check=True)
    subprocess.run(['git', 'checkout', '-q', SAM2_GIT_COMMIT], cwd=str(SAM2_DIR), check=True)
    if not (SAM2_DIR / 'setup.py').exists():
        raise RuntimeError('SAM2 clone failed: setup.py missing')
    actual_commit = _current_git_commit(SAM2_DIR)
    if actual_commit != SAM2_GIT_COMMIT:
        raise RuntimeError(f'SAM2 checkout mismatch: {actual_commit} != {SAM2_GIT_COMMIT}')

%cd {SAM2_DIR}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], check=True)


In [ ]:
#4
!pip install -q supervision[assets]

In [ ]:
#5: Verify/download the SAM2.1 Large base checkpoint used by training
from pathlib import Path

SAM2_DIR = Path(HOME) / 'segment-anything-2'
CHECKPOINT = SAM2_DIR / SAM2_BASE_CHECKPOINT_FILENAME

if CHECKPOINT.exists() and CHECKPOINT.stat().st_size > 100 * 1024 * 1024:
    print('Base checkpoint exists:', CHECKPOINT)
else:
    print('Downloading base checkpoint:', SAM2_BASE_CHECKPOINT_URL)
    !wget -q --show-progress -O {CHECKPOINT} {SAM2_BASE_CHECKPOINT_URL}
    if not CHECKPOINT.exists() or CHECKPOINT.stat().st_size < 100 * 1024 * 1024:
        raise RuntimeError(f'Base checkpoint download failed: {CHECKPOINT}')
print('Base checkpoint:', CHECKPOINT)


In [ ]:
#6
import cv2
import torch
import base64

import numpy as np
import supervision as sv

from pathlib import Path
from sam2.build_sam import build_sam2_video_predictor

IS_COLAB = True


In [ ]:
# Cell 7: Mixed precision
# Removed mixed precision for potentially higher accuracy at increased GPU load
# torch.autocast(device_type="cuda", dtype=torch.bfloat16).__enter__()

# if torch.cuda.get_device_properties(0).major >= 8:
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True


In [ ]:
# Cell 8: Prepare base and fine-tuned SAM2 predictors
import gc
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = str(Path(HOME) / 'segment-anything-2' / SAM2_BASE_CHECKPOINT_FILENAME)
CONFIG = SAM2_CONFIG_NAME
FINE_TUNED_WEIGHTS = str(FINAL_CHECKPOINT_PATH)

if not os.path.exists(FINE_TUNED_WEIGHTS):
    raise FileNotFoundError(f'Fine-tuned checkpoint not found: {FINE_TUNED_WEIGHTS}')

print('Base SAM2 checkpoint:', CHECKPOINT)
print('Fine-tuned checkpoint:', FINE_TUNED_WEIGHTS)

base_ckpt = torch.load(CHECKPOINT, map_location='cpu')
base_model_state = base_ckpt.get('model', base_ckpt) if isinstance(base_ckpt, dict) else base_ckpt
custom_ckpt = torch.load(FINE_TUNED_WEIGHTS, map_location='cpu')
FINE_TUNED_STATE_DICT = custom_ckpt.get('model', custom_ckpt) if isinstance(custom_ckpt, dict) else custom_ckpt

FINETUNED_COMPATIBLE_KEYS = []
FINETUNED_CHANGED_KEYS = []
FINETUNED_SKIPPED_SHAPE = []
FINETUNED_SKIPPED_MISSING = []
for key, value in FINE_TUNED_STATE_DICT.items():
    clean_key = key.replace('module.', '')
    base_value = base_model_state.get(clean_key)
    if base_value is None:
        FINETUNED_SKIPPED_MISSING.append(clean_key)
        continue
    if tuple(base_value.shape) != tuple(value.shape):
        FINETUNED_SKIPPED_SHAPE.append(clean_key)
        continue
    FINETUNED_COMPATIBLE_KEYS.append(clean_key)
    if not torch.equal(base_value.cpu(), value.cpu()):
        FINETUNED_CHANGED_KEYS.append(clean_key)

if not FINETUNED_COMPATIBLE_KEYS:
    raise RuntimeError('No compatible fine-tuned parameters match the SAM2 base checkpoint.')
if not FINETUNED_CHANGED_KEYS:
    raise RuntimeError('Fine-tuned checkpoint is compatible but does not differ from the base checkpoint.')

print('Checkpoint comparison')
print(f'  Base model keys:              {len(base_model_state)}')
print(f'  Fine-tuned model keys:        {len(FINE_TUNED_STATE_DICT)}')
print(f'  Compatible keys:              {len(FINETUNED_COMPATIBLE_KEYS)}')
print(f'  Compatible keys changed:      {len(FINETUNED_CHANGED_KEYS)}')
print(f'  Skipped shape mismatches:     {len(FINETUNED_SKIPPED_SHAPE)}')
print(f'  Skipped missing in base:      {len(FINETUNED_SKIPPED_MISSING)}')

# Keep only the fine-tuned state dict for later sequential predictor builds.
del base_ckpt, base_model_state, custom_ckpt
gc.collect()

def build_sam2_predictor_variant(model_variant: str):
    """Build one predictor variant. Variants are run sequentially to conserve GPU memory."""
    model_variant = str(model_variant).strip().lower()
    predictor = build_sam2_video_predictor(CONFIG, CHECKPOINT)
    load_report = {
        'model_variant': model_variant,
        'base_checkpoint': CHECKPOINT,
        'fine_tuned_checkpoint': FINE_TUNED_WEIGHTS,
        'loaded_compatible_keys': 0,
        'missing_after_load': 0,
        'unexpected_after_load': 0,
        'skipped_shape': 0,
        'skipped_missing': 0,
    }
    if model_variant == 'base':
        predictor.eval()
        return predictor, load_report
    if model_variant != 'finetuned':
        raise ValueError(f'Unknown model variant: {model_variant!r}')

    current_state = predictor.state_dict()
    compatible_state = {}
    skipped_shape = []
    skipped_missing = []
    for key, value in FINE_TUNED_STATE_DICT.items():
        clean_key = key.replace('module.', '')
        if clean_key in current_state:
            if tuple(current_state[clean_key].shape) == tuple(value.shape):
                compatible_state[clean_key] = value
            else:
                skipped_shape.append(clean_key)
        else:
            skipped_missing.append(clean_key)
    if not compatible_state:
        raise RuntimeError('No compatible fine-tuned parameters matched the SAM2 video predictor.')

    result = predictor.load_state_dict(compatible_state, strict=False)
    predictor.eval()
    load_report.update({
        'loaded_compatible_keys': len(compatible_state),
        'missing_after_load': len(result.missing_keys),
        'unexpected_after_load': len(result.unexpected_keys),
        'skipped_shape': len(skipped_shape),
        'skipped_missing': len(skipped_missing),
    })
    print(f'Loaded fine-tuned predictor: {len(compatible_state)} compatible parameters')
    return predictor, load_report


In [ ]:
# Cell 8.5: Checkpoint summary
print('Checkpoint summary')
print(f'  Base SAM2.1 Large path:       {CHECKPOINT}')
print(f'  Fine-tuned path:              {FINE_TUNED_WEIGHTS}')
print(f'  Compatible fine-tuned keys:   {len(FINETUNED_COMPATIBLE_KEYS)}')
print(f'  Changed compatible keys:      {len(FINETUNED_CHANGED_KEYS)}')
print('  This notebook runs explicit model variants; base and fine-tuned outputs are saved separately for every manifest clip.')


In [ ]:
# Cell 9: Frame-extraction parameters
# These values match the prompt-coordinate frame produced by the prompt-manifest
# notebook: the per-clip prompt points and boxes in the transfer manifest are recorded in the
# clip's native pixel coordinates, and TARGET_LONG_SIDE only down-scales when a
# frame's long side exceeds 1024 px. The reviewed clips are well below that
# threshold, so the extracted frames keep the prompt-frame coordinates exactly.
START_IDX = 0
END_IDX = None        # None means through the end of each clip.
FRAME_STRIDE = 1
TARGET_LONG_SIDE = 1024


In [ ]:
# Cell 10: Load the transfer manifest and pick clip indices
import json
from pathlib import Path

if not E06_TRANSFER_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f'Transfer manifest not found: {E06_TRANSFER_MANIFEST_PATH}. '
        'Run the prompt-manifest notebook to write the multi-clip transfer manifest first.'
    )

E06_TRANSFER_MANIFEST = json.loads(E06_TRANSFER_MANIFEST_PATH.read_text(encoding='utf-8'))
ALL_CLIP_ENTRIES = E06_TRANSFER_MANIFEST.get('clips') or []
if not ALL_CLIP_ENTRIES:
    raise RuntimeError(
        f'Transfer manifest has no clips entries: {E06_TRANSFER_MANIFEST_PATH}'
    )

print(f'Loaded transfer manifest with {len(ALL_CLIP_ENTRIES)} clip(s).')


def resolve_supplemental_path(value) -> Path:
    """Resolve a path that may be supplemental-relative or absolute."""
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path
    candidates = [
        SUPPLEMENTAL_ROOT / path,
        SUPPLEMENTAL_ROOT.parent / path,
        Path.cwd() / path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    # Return the first candidate so the caller sees a sensible error path.
    return candidates[0]


def _parse_clip_index_override(raw) -> list[int]:
    if raw is None or raw == '':
        return []
    if isinstance(raw, int):
        return [int(raw)]
    if isinstance(raw, (list, tuple)):
        return [int(x) for x in raw]
    tokens = [tok.strip() for tok in str(raw).split(',') if tok.strip() != '']
    return [int(tok) for tok in tokens]


_clip_index_override_raw = (
    globals().get('SYNISCOPY_E09_CLIP_INDEX', None)
    or os.environ.get('SYNISCOPY_E09_CLIP_INDEX')
)
SELECTED_CLIP_INDICES = _parse_clip_index_override(_clip_index_override_raw)
if SELECTED_CLIP_INDICES:
    available = {int(entry.get('clip_index', idx)) for idx, entry in enumerate(ALL_CLIP_ENTRIES)}
    missing = sorted(set(SELECTED_CLIP_INDICES) - available)
    if missing:
        raise ValueError(
            f'SYNISCOPY_E09_CLIP_INDEX includes clip indices not present in the transfer manifest: {missing}'
        )
    selected_entries = [
        entry for idx, entry in enumerate(ALL_CLIP_ENTRIES)
        if int(entry.get('clip_index', idx)) in set(SELECTED_CLIP_INDICES)
    ]
    print(f'SYNISCOPY_E09_CLIP_INDEX override active: running {len(selected_entries)} clip(s) -> {SELECTED_CLIP_INDICES}')
else:
    selected_entries = list(ALL_CLIP_ENTRIES)
    print(f'No SYNISCOPY_E09_CLIP_INDEX override; running all {len(selected_entries)} clip(s).')

CLIP_ENTRIES = selected_entries


In [ ]:
# Cell 11: Variants, objects, and review thresholds
OBJECTS = ['particle']

# This is the paper real-video transfer-inference copy. It saves both base and fine-tuned
# SAM2 outputs so the transfer artifact can be checked directly.
MODEL_VARIANTS_TO_RUN = ['base', 'finetuned']
# Save both variants for direct comparison. The fine-tuned variant remains the
# primary named output because this notebook is evaluating the trained checkpoint, not hiding
# it.
PRIMARY_OUTPUT_VARIANT = 'finetuned'

# These are review thresholds only. They write warnings into each variant's
# mask_metrics.json; they do not stop the notebook.
MASK_REVIEW_WARN_FRACTION = 0.25
MASK_REVIEW_LARGE_FRAME_FRACTION = 0.60


In [ ]:
# Cell 12: Per-clip helpers (frame extraction, prompt manifest, variant runner)
import json
import shutil
import subprocess
from pathlib import Path

import cv2
import numpy as np
import supervision as sv


def read_video_info(video_path: Path) -> sv.VideoInfo:
    try:
        return sv.VideoInfo.from_video_path(str(video_path))
    except Exception:
        pass
    cap = cv2.VideoCapture(str(video_path))
    if cap.isOpened():
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        if width > 0 and height > 0 and fps > 0 and total > 0:
            return sv.VideoInfo(width=width, height=height, fps=fps, total_frames=total)
    else:
        cap.release()
    try:
        raw = subprocess.check_output(
            [
                'ffprobe', '-v', 'error', '-select_streams', 'v:0',
                '-show_entries', 'stream=width,height,nb_frames,r_frame_rate,duration',
                '-of', 'json', str(video_path),
            ],
            text=True,
        )
        stream = (json.loads(raw).get('streams') or [{}])[0]
        width = int(stream.get('width') or 0)
        height = int(stream.get('height') or 0)
        total = int(stream.get('nb_frames') or 0)
        rate = stream.get('r_frame_rate') or '0/0'
        num, den = rate.split('/')
        fps = float(num) / float(den) if float(den) else 0.0
        if total <= 0 and stream.get('duration') and fps > 0:
            total = int(round(float(stream['duration']) * fps))
        if width > 0 and height > 0 and fps > 0 and total > 0:
            return sv.VideoInfo(width=width, height=height, fps=fps, total_frames=total)
    except Exception as exc:
        print('ffprobe video metadata fallback failed:', repr(exc))
    raise RuntimeError(f'Could not read video metadata for {video_path}')


def extract_frames_for_clip(source_video: Path, source_frames_dir: Path) -> tuple[int, float, int]:
    """Decode clip frames to JPEGs in source_frames_dir using the preserved
    TARGET_LONG_SIDE down-scale rule. Returns (written_count, source_fps, source_total)."""
    source_frames_dir.mkdir(parents=True, exist_ok=True)
    for old in source_frames_dir.glob('*.jpeg'):
        old.unlink()

    cap = cv2.VideoCapture(str(source_video))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else 0
    fps_value = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else 0.0
    start = max(0, int(START_IDX))
    stride = max(1, int(FRAME_STRIDE))
    if total <= 0:
        end = None if END_IDX is None else int(END_IDX)
    else:
        end = total if END_IDX is None else min(total, int(END_IDX))
    if end is not None and start >= end:
        raise ValueError(
            f'Invalid frame range for {source_video.name}: '
            f'START_IDX={START_IDX}, END_IDX={END_IDX}, decoded total={total}'
        )

    def _write_frame(frame_bgr, out_index: int) -> None:
        h, w = frame_bgr.shape[:2]
        scale = min(1.0, float(TARGET_LONG_SIDE) / max(h, w)) if TARGET_LONG_SIDE else 1.0
        if scale != 1.0:
            new_w = max(1, int(round(w * scale)))
            new_h = max(1, int(round(h * scale)))
            frame_bgr = cv2.resize(frame_bgr, (new_w, new_h), interpolation=cv2.INTER_AREA)
        out_path = source_frames_dir / f'{out_index:05d}.jpeg'
        cv2.imwrite(str(out_path), frame_bgr)

    written = 0
    if cap.isOpened():
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
        source_idx = start
        while end is None or source_idx < end:
            ret, frame_bgr = cap.read()
            if not ret:
                break
            if (source_idx - start) % stride == 0:
                _write_frame(frame_bgr, written)
                written += 1
            source_idx += 1
        cap.release()
    else:
        cap.release()
        raw_frame_dir = source_frames_dir.parent / f'{source_video.stem}_ffmpeg_frames'
        if raw_frame_dir.exists():
            shutil.rmtree(raw_frame_dir)
        raw_frame_dir.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(source_video), '-q:v', '2', str(raw_frame_dir / '%05d.jpeg')],
            check=True,
        )
        raw_paths = sorted(raw_frame_dir.glob('*.jpeg'))
        total = len(raw_paths)
        stop = total if end is None else min(total, end)
        for raw_path in raw_paths[start:stop:stride]:
            frame_bgr = cv2.imread(str(raw_path), cv2.IMREAD_COLOR)
            if frame_bgr is None:
                continue
            _write_frame(frame_bgr, written)
            written += 1

    if written == 0:
        raise RuntimeError(f'Frame extraction wrote zero frames for {source_video}')
    return written, fps_value, total


def point_prompt_label(point) -> int:
    return int(point.get('point_label', point.get('prompt_label', 1)))


def box_to_xyxy(box, frame_width: int, frame_height: int):
    if box is None:
        return None
    if 'xyxy' in box:
        x1, y1, x2, y2 = box['xyxy']
    elif all(k in box for k in ('x1', 'y1', 'x2', 'y2')):
        x1, y1, x2, y2 = box['x1'], box['y1'], box['x2'], box['y2']
    elif all(k in box for k in ('x', 'y', 'width', 'height')):
        # Legacy box dictionaries use top-left x/y with width/height.
        x1 = box['x']
        y1 = box['y']
        x2 = box['x'] + box['width']
        y2 = box['y'] + box['height']
    else:
        raise ValueError(f'Unsupported box format: {box}')
    x1 = float(np.clip(x1, 0, frame_width - 1))
    y1 = float(np.clip(y1, 0, frame_height - 1))
    x2 = float(np.clip(x2, 0, frame_width - 1))
    y2 = float(np.clip(y2, 0, frame_height - 1))
    if x2 <= x1 or y2 <= y1:
        raise ValueError(f'Invalid clipped box: {(x1, y1, x2, y2)} from {box}')
    return [x1, y1, x2, y2]


def prompt_arrays(points_list, label: str):
    selected = [p for p in points_list if p.get('label') == label]
    points = np.array([[p['x'], p['y']] for p in selected], dtype=np.float32)
    labels = np.array([point_prompt_label(p) for p in selected], dtype=np.int32)
    return points, labels


def build_clip_prompt_manifest(
    clip_entry: dict,
    source_video: Path,
    source_frames_dir: Path,
    source_frame_paths: list[str],
    run_output_dir: Path,
) -> tuple[Path, dict, dict, dict, int, int]:
    """Write the per-clip prompt_manifest.json and return prompt structures."""
    sample = cv2.imread(source_frame_paths[0])
    if sample is None:
        raise RuntimeError(f'Could not read first extracted frame: {source_frame_paths[0]}')
    frame_height, frame_width = sample.shape[:2]

    frame_points_by_idx: dict[int, list[dict]] = {}
    frame_boxes_by_idx: dict[int, list[dict]] = {}
    for prompt_entry in clip_entry.get('prompts', []):
        idx = int(prompt_entry.get('frame_idx', 0))
        points = [dict(point) for point in prompt_entry.get('points', [])]
        boxes = [dict(box) for box in prompt_entry.get('boxes', [])]
        if not points:
            continue
        frame_points_by_idx.setdefault(idx, []).extend(points)
        if boxes:
            frame_boxes_by_idx.setdefault(idx, []).extend(boxes)

    if not frame_points_by_idx:
        raise RuntimeError(
            f'No prompt points are defined for clip {source_video.name} in the transfer manifest.'
        )

    prompt_manifest = {
        'schema_version': 'syniscopy-sam2-prompt-v2',
        'source_video': str(source_video),
        'source_frames': str(source_frames_dir),
        'frame_count': len(source_frame_paths),
        'frame_width': int(frame_width),
        'frame_height': int(frame_height),
        'model_variants_to_run': list(MODEL_VARIANTS_TO_RUN),
        'use_fixed_prompt': True,
        'prompt_source': 'supplemental/outputs/E06/sam2-caustic-transfer-manifest.json',
        'e06_clip_index': int(clip_entry.get('clip_index', -1)),
        'e06_clip_video': clip_entry.get('clip_video'),
        'source_video_original': clip_entry.get('source_video'),
        'source_frame_start': clip_entry.get('source_frame_start'),
        'source_frame_end_exclusive': clip_entry.get('source_frame_end_exclusive'),
        'source_prompt_frame_idx': clip_entry.get('source_prompt_frame_idx'),
        'fps_metadata': clip_entry.get('fps'),
        'fps_expected_from_repository': clip_entry.get('fps_expected_from_repository'),
        'analysis_fps': clip_entry.get('fps_expected_from_repository') or clip_entry.get('fps'),
        'pixel_size_nm': clip_entry.get('pixel_size_nm'),
        'particle_diameter_nm': clip_entry.get('particle_diameter_nm'),
        'prompts': [],
    }
    for idx in sorted(set(frame_points_by_idx) | set(frame_boxes_by_idx)):
        boxes = [dict(box) for box in frame_boxes_by_idx.get(idx, [])]
        prompt_manifest['prompts'].append({
            'frame_idx': int(idx),
            'frame_path': str(source_frames_dir / f'{int(idx):05d}.jpeg'),
            'points': [dict(point) for point in frame_points_by_idx.get(idx, [])],
            'boxes': boxes,
            'box_xyxy': [box_to_xyxy(box, frame_width, frame_height) for box in boxes],
        })

    prompt_manifest_path = run_output_dir / 'prompt_manifest.json'
    run_output_dir.mkdir(parents=True, exist_ok=True)
    with open(prompt_manifest_path, 'w', encoding='utf-8') as fh:
        json.dump(prompt_manifest, fh, indent=2, sort_keys=True)

    return (
        prompt_manifest_path,
        prompt_manifest,
        frame_points_by_idx,
        frame_boxes_by_idx,
        int(frame_width),
        int(frame_height),
    )


def add_prompt_to_predictor(predictor, inference_state, frame_idx, object_id, points, labels, box_prompt=None):
    kwargs = dict(
        inference_state=inference_state,
        frame_idx=int(frame_idx),
        obj_id=int(object_id),
        points=points,
        labels=labels,
    )
    if box_prompt is not None:
        kwargs['box'] = box_prompt
    if hasattr(predictor, 'add_new_points_or_box'):
        return predictor.add_new_points_or_box(**kwargs)
    return predictor.add_new_points(**kwargs)


def add_all_prompts_to_predictor(
    predictor,
    inference_state,
    frame_points_by_idx: dict[int, list[dict]],
    frame_boxes_by_idx: dict[int, list[dict]],
    frame_width: int,
    frame_height: int,
) -> list[dict]:
    added: list[dict] = []
    for frame_idx, points_list in frame_points_by_idx.items():
        for object_id, label in enumerate(OBJECTS, start=1):
            points, labels = prompt_arrays(points_list, label)
            if len(points) == 0:
                continue
            box_prompt = None
            if frame_idx in frame_boxes_by_idx:
                bbox_objects = [bbox for bbox in frame_boxes_by_idx[frame_idx] if bbox.get('label') == label]
                if bbox_objects:
                    box_prompt = np.array(
                        box_to_xyxy(bbox_objects[0], frame_width, frame_height),
                        dtype=np.float32,
                    )
            try:
                add_prompt_to_predictor(
                    predictor, inference_state, frame_idx, object_id, points, labels, box_prompt=box_prompt
                )
            except TypeError:
                if box_prompt is None:
                    raise
                box_points = np.array(
                    [[box_prompt[0], box_prompt[1]], [box_prompt[2], box_prompt[3]]],
                    dtype=np.float32,
                )
                all_points = np.vstack([points, box_points])
                all_labels = np.concatenate([labels, np.array([2, 3], dtype=np.int32)])
                add_prompt_to_predictor(
                    predictor, inference_state, frame_idx, object_id, all_points, all_labels, box_prompt=None
                )
            added.append({
                'frame_idx': int(frame_idx),
                'object_id': int(object_id),
                'label': label,
                'point_count': int(len(points)),
                'positive_points': int(np.count_nonzero(labels == 1)),
                'negative_points': int(np.count_nonzero(labels == 0)),
                'box_used': box_prompt is not None,
            })
    if not added:
        raise RuntimeError('No prompts were added to the predictor.')
    return added


def variant_paths(run_output_dir: Path, source_video: Path, model_variant: str) -> dict:
    variant_dir = run_output_dir / model_variant
    mask_dir = variant_dir / 'masks'
    variant_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)
    stem = source_video.stem
    return {
        'variant_dir': variant_dir,
        'overlay_video': variant_dir / f'{stem}_{model_variant}_overlay.avi',
        'mask_only_video': variant_dir / f'{stem}_{model_variant}_mask_only.avi',
        'mask_frame_dir': mask_dir,
        'metrics_json': variant_dir / 'mask_metrics.json',
    }


def variant_outputs_complete(paths: dict) -> tuple[bool, str | None]:
    """Return (complete, reason_for_rerun_if_incomplete)."""
    overlay = paths['overlay_video']
    mask_only = paths['mask_only_video']
    metrics_json = paths['metrics_json']
    mask_dir = paths['mask_frame_dir']
    if not overlay.exists() or overlay.stat().st_size == 0:
        return False, f'overlay missing or empty: {overlay.name}'
    if not mask_only.exists() or mask_only.stat().st_size == 0:
        return False, f'mask-only video missing or empty: {mask_only.name}'
    if not metrics_json.exists() or metrics_json.stat().st_size == 0:
        return False, f'metrics missing or empty: {metrics_json.name}'
    try:
        metrics = json.loads(metrics_json.read_text(encoding='utf-8'))
    except json.JSONDecodeError as exc:
        return False, f'metrics not parseable: {exc}'
    if not isinstance(metrics, dict):
        return False, 'metrics is not a JSON object'
    frame_count = int(metrics.get('frame_count_written', 0))
    if frame_count <= 0:
        return False, 'metrics frame_count_written is 0'
    png_masks = list(mask_dir.glob('frame_*_object_*.png'))
    if not png_masks:
        return False, 'no per-frame PNG masks present'
    # Compare loosely: SAM2 may produce one PNG per (frame, object), so allow >= frame_count.
    if len(png_masks) < frame_count:
        return False, (
            f'per-frame PNG count {len(png_masks)} is less than metrics frame_count_written {frame_count}'
        )
    return True, None


def _object_ids_array(object_ids):
    if hasattr(object_ids, 'detach'):
        return object_ids.detach().cpu().numpy().astype(int)
    return np.array(object_ids, dtype=int)


def run_sam2_variant_for_clip(
    model_variant: str,
    *,
    source_video: Path,
    source_frames_dir: Path,
    source_frame_paths: list[str],
    run_output_dir: Path,
    prompt_manifest_path: Path,
    frame_points_by_idx: dict[int, list[dict]],
    frame_boxes_by_idx: dict[int, list[dict]],
    frame_width: int,
    frame_height: int,
    video_info: sv.VideoInfo,
    mask_annotator: sv.MaskAnnotator,
) -> dict:
    paths = variant_paths(run_output_dir, source_video, model_variant)
    complete, reason = variant_outputs_complete(paths)
    if complete:
        metrics = json.loads(paths['metrics_json'].read_text(encoding='utf-8'))
        print(f'  [{model_variant}] complete -> skip (reusing existing outputs)')
        return {
            'status': 'skipped_complete',
            'overlay_video': str(paths['overlay_video']),
            'mask_only_video': str(paths['mask_only_video']),
            'mask_frame_dir': str(paths['mask_frame_dir']),
            'metrics_json': str(paths['metrics_json']),
            'metrics': metrics,
        }
    print(f'  [{model_variant}] running (reason: {reason})')

    for old_mask in paths['mask_frame_dir'].glob('*.png'):
        old_mask.unlink()

    predictor, load_report = build_sam2_predictor_variant(model_variant)
    inference_state = predictor.init_state(video_path=source_frames_dir.as_posix())
    predictor.reset_state(inference_state)
    prompt_report = add_all_prompts_to_predictor(
        predictor,
        inference_state,
        frame_points_by_idx,
        frame_boxes_by_idx,
        frame_width,
        frame_height,
    )

    frame_count = 0
    error_count = 0
    area_fractions: list[float] = []
    per_frame: list[dict] = []

    with sv.VideoSink(paths['overlay_video'].as_posix(), video_info=video_info) as overlay_sink, \
         sv.VideoSink(paths['mask_only_video'].as_posix(), video_info=video_info) as mask_sink:
        for frame_idx, object_ids, mask_logits in predictor.propagate_in_video(inference_state):
            try:
                frame_path = source_frame_paths[frame_idx]
                frame = cv2.imread(str(frame_path))
                if frame is None:
                    raise RuntimeError(f'Failed to read frame {frame_idx}: {frame_path}')
                frame_h, frame_w = frame.shape[:2]
                masks = (mask_logits > 0.0).cpu().numpy().astype(bool)
                if masks.ndim == 4:
                    masks = masks.squeeze(1)
                resized_masks = np.zeros((masks.shape[0], frame_h, frame_w), dtype=bool)
                mask_frame = np.zeros((frame_h, frame_w, 3), dtype=np.uint8)
                object_ids_np = _object_ids_array(object_ids)

                frame_areas: list[float] = []
                for i in range(masks.shape[0]):
                    resized_mask = cv2.resize(
                        masks[i].astype(np.uint8),
                        (frame_w, frame_h),
                        interpolation=cv2.INTER_NEAREST,
                    ).astype(bool)
                    resized_masks[i] = resized_mask
                    mask_frame[resized_mask] = [255, 255, 255]
                    object_id = int(object_ids_np[i]) if i < len(object_ids_np) else i + 1
                    cv2.imwrite(
                        str(paths['mask_frame_dir'] / f'frame_{frame_idx:05d}_object_{object_id}.png'),
                        resized_mask.astype(np.uint8) * 255,
                    )
                    frac = float(np.count_nonzero(resized_mask) / resized_mask.size)
                    frame_areas.append(frac)
                    area_fractions.append(frac)

                detections = sv.Detections(
                    xyxy=sv.mask_to_xyxy(masks=resized_masks),
                    mask=resized_masks,
                    class_id=object_ids_np,
                )
                annotated_frame = mask_annotator.annotate(scene=frame.copy(), detections=detections)
                overlay_sink.write_frame(annotated_frame)
                mask_sink.write_frame(mask_frame)
                frame_count += 1
                per_frame.append({
                    'frame_idx': int(frame_idx),
                    'object_ids': [int(x) for x in object_ids_np.tolist()],
                    'mask_area_fraction_max': max(frame_areas) if frame_areas else 0.0,
                    'mask_area_fraction_sum': float(sum(frame_areas)),
                })
                if frame_idx % 50 == 0:
                    print(f'     frame {frame_idx}/{len(source_frame_paths)}')
            except Exception as exc:
                print(f'     error on frame {frame_idx}: {exc}')
                error_count += 1
                continue

    if frame_count == 0:
        raise RuntimeError(f'{model_variant} propagation wrote zero frames for {source_video.name}')
    if not paths['overlay_video'].exists() or paths['overlay_video'].stat().st_size == 0:
        raise RuntimeError(f'Overlay video was not written: {paths["overlay_video"]}')
    if not paths['mask_only_video'].exists() or paths['mask_only_video'].stat().st_size == 0:
        raise RuntimeError(f'Mask-only video was not written: {paths["mask_only_video"]}')

    max_area = max(area_fractions) if area_fractions else 0.0
    median_area = float(np.median(area_fractions)) if area_fractions else 0.0
    review_warnings: list[str] = []
    if median_area > MASK_REVIEW_WARN_FRACTION:
        review_warnings.append(
            f'median mask area fraction {median_area:.3f} exceeds review threshold {MASK_REVIEW_WARN_FRACTION:.3f}'
        )
    if max_area > MASK_REVIEW_LARGE_FRAME_FRACTION:
        review_warnings.append(
            f'max mask area fraction {max_area:.3f} exceeds large-frame threshold {MASK_REVIEW_LARGE_FRAME_FRACTION:.3f}'
        )

    metrics = {
        'schema_version': 'syniscopy-sam2-inference-metrics-v1',
        'model_variant': model_variant,
        'source_video': str(source_video),
        'source_frames': str(source_frames_dir),
        'frame_count_written': int(frame_count),
        'error_count': int(error_count),
        'max_mask_area_fraction': float(max_area),
        'median_mask_area_fraction': float(median_area),
        'review_warnings': review_warnings,
        'prompt_manifest': str(prompt_manifest_path),
        'load_report': load_report,
        'prompt_report': prompt_report,
        'per_frame': per_frame,
    }
    with open(paths['metrics_json'], 'w', encoding='utf-8') as fh:
        json.dump(metrics, fh, indent=2)

    print(f'  [{model_variant}] wrote {frame_count} frames, errors={error_count}')
    if review_warnings:
        for warning in review_warnings:
            print('     warning:', warning)

    del predictor, inference_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'status': 'completed',
        'overlay_video': str(paths['overlay_video']),
        'mask_only_video': str(paths['mask_only_video']),
        'mask_frame_dir': str(paths['mask_frame_dir']),
        'metrics_json': str(paths['metrics_json']),
        'metrics': metrics,
    }


def supplemental_relative(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(SUPPLEMENTAL_ROOT.resolve()))
    except Exception:
        try:
            return str(path.resolve().relative_to(SUPPLEMENTAL_ROOT.parent.resolve()))
        except Exception:
            return str(path)


In [ ]:
# Cell 13: Iterate clips, run variants, write the batch manifest
import json
from pathlib import Path

import cv2
import supervision as sv

COLORS = ['#FF1493']
MASK_ANNOTATOR = sv.MaskAnnotator(color=sv.ColorPalette.from_hex(COLORS), color_lookup=sv.ColorLookup.CLASS)

BATCH_MANIFEST_PATH = INFERENCE_OUTPUT_ROOT / 'e09_batch_manifest.json'
INFERENCE_RESULTS: dict[str, dict] = {}
batch_records: list[dict] = []
batch_warnings: list[str] = []

for clip_position, clip_entry in enumerate(CLIP_ENTRIES):
    clip_index = int(clip_entry.get('clip_index', clip_position))
    clip_video_rel = clip_entry.get('clip_video')
    if not clip_video_rel:
        raise RuntimeError(f'Transfer-manifest clip entry {clip_index} is missing clip_video')
    source_video = resolve_supplemental_path(clip_video_rel)
    if not source_video.exists():
        raise FileNotFoundError(
            f'Transfer clip video not found for clip_index={clip_index}: {source_video}'
        )

    clip_stem = source_video.stem
    print(f'\n=== Clip {clip_index:02d}: {clip_stem} ===')
    print(f'    source: {source_video}')

    source_frames_dir = Path(HOME) / f'{clip_stem}_frames'
    run_output_dir = INFERENCE_OUTPUT_ROOT / clip_stem
    run_output_dir.mkdir(parents=True, exist_ok=True)

    # Always re-extract frames here. Extraction is cheap relative to SAM2
    # propagation, and the per-clip frames dir lives on the Colab VM (not
    # Drive), so reuse across reruns is not assumed to be reliable.
    extracted, source_fps_decoded, source_total = extract_frames_for_clip(
        source_video, source_frames_dir
    )
    source_frame_paths = sorted(str(p) for p in source_frames_dir.glob('*.jpeg'))
    if not source_frame_paths:
        raise RuntimeError(f'No extracted frames found in {source_frames_dir}')

    fps_value = float(source_fps_decoded or 0.0)
    if fps_value <= 0:
        fps_value = float(clip_entry.get('fps') or 0.0)
    if fps_value <= 0:
        fps_value = 5.0
    analysis_fps = float(clip_entry.get('fps_expected_from_repository') or 0.0)
    if analysis_fps <= 0:
        analysis_fps = float(clip_entry.get('fps') or fps_value)
    pixel_size_nm = clip_entry.get('pixel_size_nm')

    prompt_manifest_path, prompt_manifest, frame_points_by_idx, frame_boxes_by_idx, frame_width, frame_height = (
        build_clip_prompt_manifest(
            clip_entry,
            source_video,
            source_frames_dir,
            source_frame_paths,
            run_output_dir,
        )
    )

    video_info = sv.VideoInfo(
        width=int(frame_width),
        height=int(frame_height),
        fps=fps_value,
        total_frames=len(source_frame_paths),
    )
    print(
        f'    extracted {extracted} frame(s) at {frame_width}x{frame_height} '
        f'@ {fps_value:.3f} fps playback; analysis fps {analysis_fps:.3f}; prompt frames: '
        f'{sorted(set(frame_points_by_idx) | set(frame_boxes_by_idx))}'
    )

    clip_variant_results: dict[str, dict] = {}
    clip_warnings: list[str] = []
    for variant in MODEL_VARIANTS_TO_RUN:
        variant_result = run_sam2_variant_for_clip(
            str(variant),
            source_video=source_video,
            source_frames_dir=source_frames_dir,
            source_frame_paths=source_frame_paths,
            run_output_dir=run_output_dir,
            prompt_manifest_path=prompt_manifest_path,
            frame_points_by_idx=frame_points_by_idx,
            frame_boxes_by_idx=frame_boxes_by_idx,
            frame_width=frame_width,
            frame_height=frame_height,
            video_info=video_info,
            mask_annotator=MASK_ANNOTATOR,
        )
        clip_variant_results[str(variant)] = variant_result
        for warning in variant_result.get('metrics', {}).get('review_warnings', []) or []:
            clip_warnings.append(f'{variant}: {warning}')

    INFERENCE_RESULTS[clip_stem] = clip_variant_results

    batch_records.append({
        'clip_index': clip_index,
        'clip_stem': clip_stem,
        'clip_video': clip_entry.get('clip_video'),
        'source_video_original': clip_entry.get('source_video'),
        'source_video_resolved': str(source_video),
        'audit_json': clip_entry.get('audit_json'),
        'source_frame_start': clip_entry.get('source_frame_start'),
        'source_frame_end_exclusive': clip_entry.get('source_frame_end_exclusive'),
        'source_prompt_frame_idx': clip_entry.get('source_prompt_frame_idx'),
        'prompt_source': supplemental_relative(E06_TRANSFER_MANIFEST_PATH),
        'prompt_manifest': supplemental_relative(prompt_manifest_path),
        'run_output_dir': supplemental_relative(run_output_dir),
        'frame_count_extracted': int(extracted),
        'frame_width': int(frame_width),
        'frame_height': int(frame_height),
        'fps': float(fps_value),
        'fps_metadata': float(fps_value),
        'fps_expected_from_repository': float(analysis_fps),
        'analysis_fps': float(analysis_fps),
        'pixel_size_nm': float(pixel_size_nm) if pixel_size_nm is not None else None,
        'particle_diameter_nm': clip_entry.get('particle_diameter_nm'),
        'model_variants_to_run': list(MODEL_VARIANTS_TO_RUN),
        'variants': {
            variant: {
                'status': result['status'],
                'overlay_video': supplemental_relative(Path(result['overlay_video'])),
                'mask_only_video': supplemental_relative(Path(result['mask_only_video'])),
                'mask_frame_dir': supplemental_relative(Path(result['mask_frame_dir'])),
                'metrics_json': supplemental_relative(Path(result['metrics_json'])),
                'frame_count_written': int(
                    result.get('metrics', {}).get('frame_count_written', 0)
                ),
                'error_count': int(result.get('metrics', {}).get('error_count', 0)),
                'review_warnings': list(
                    result.get('metrics', {}).get('review_warnings', []) or []
                ),
                'load_report': dict(result.get('metrics', {}).get('load_report', {}) or {}),
            }
            for variant, result in clip_variant_results.items()
        },
        'warnings': clip_warnings,
        'completion_status': (
            'complete'
            if all(
                result['status'] in {'completed', 'skipped_complete'}
                for result in clip_variant_results.values()
            )
            else 'incomplete'
        ),
    })
    for warning in clip_warnings:
        batch_warnings.append(f'{clip_stem}: {warning}')

batch_manifest = {
    'schema_version': 'syniscopy-e09-batch-manifest-v1',
    'notebook_version': RELEASE_NOTEBOOK_VERSION,
    'e06_transfer_manifest': supplemental_relative(E06_TRANSFER_MANIFEST_PATH),
    'e06_clip_count': int(len(ALL_CLIP_ENTRIES)),
    'clip_count_run': int(len(batch_records)),
    'selected_clip_indices': list(SELECTED_CLIP_INDICES) if SELECTED_CLIP_INDICES else None,
    'model_variants_to_run': list(MODEL_VARIANTS_TO_RUN),
    'primary_output_variant': PRIMARY_OUTPUT_VARIANT,
    'inference_output_root': supplemental_relative(INFERENCE_OUTPUT_ROOT),
    'clips': batch_records,
    'warnings': batch_warnings,
}

with open(BATCH_MANIFEST_PATH, 'w', encoding='utf-8') as fh:
    json.dump(batch_manifest, fh, indent=2, sort_keys=True)

print('\nBatch manifest written:', BATCH_MANIFEST_PATH)
print(f'Clips run: {len(batch_records)} (out of {len(ALL_CLIP_ENTRIES)} in transfer manifest)')
if batch_warnings:
    print('Review warnings present:')
    for warning in batch_warnings:
        print('  -', warning)


In [ ]:
# Cell 14: Per-clip summary of output artifacts
print('Inference output root:', INFERENCE_OUTPUT_ROOT)
print('Batch manifest:        ', BATCH_MANIFEST_PATH)
print()
for record in batch_records:
    print(f'[{record["clip_index"]:02d}] {record["clip_stem"]} ({record["completion_status"]})')
    print('    prompt manifest:', record['prompt_manifest'])
    for variant, info in record['variants'].items():
        print(
            f'    {variant:>9}: status={info["status"]}, frames={info["frame_count_written"]}, '
            f'errors={info["error_count"]}'
        )
        print('              overlay   :', info['overlay_video'])
        print('              mask-only :', info['mask_only_video'])
        if info['review_warnings']:
            print('              warnings  :', info['review_warnings'])
